# Global Trends

### Imports and preprocessing

In [1]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import linregress
from pathlib import Path

ROOT_DIR = Path.cwd().parent
Tables_DIR = ROOT_DIR / "data/tables"

# helper that saves each graph + summary so src/make_pdf.py can build the PDF
sys.path.insert(0, str(ROOT_DIR / "src"))
from report import save_report

TOPIC = "global_trends"

monthlyDf = pd.read_csv(Tables_DIR / "global_monthly.csv")
yearlyDf = pd.read_csv(Tables_DIR / "global_yearly.csv")

# sort once so every question works on time-ordered data
yearlyDf = yearlyDf.sort_values("year").reset_index(drop=True)
monthlyDf = monthlyDf.sort_values(["year", "month"]).reset_index(drop=True)

print("yearly :", yearlyDf.shape, "| years", int(yearlyDf["year"].min()), "-", int(yearlyDf["year"].max()))
print("monthly:", monthlyDf.shape)

yearly : (147, 16) | years 1880 - 2026
monthly: (1760, 8)


### Q1. How has CO2 changed from 1960 to now?

In [2]:
a1 = yearlyDf.dropna(subset=["co2_mlo_yoy_change"])
g1 = a1[["year", "co2_mlo_yoy_change"]].rename(columns={"co2_mlo_yoy_change": "co2_ppm"})

# Rebuild CO2 level: base (1960) + running total of yearly growth
BASE_1960 = 316.91   # ppm, NOAA Mauna Loa annual mean for 1960
g1 = g1.sort_values("year").reset_index(drop=True)
g1["co2_level"] = BASE_1960 + g1["co2_ppm"].cumsum() - g1["co2_ppm"].iloc[0]

first, last = g1.iloc[0], g1.iloc[-1]
total_rise = last["co2_level"] - first["co2_level"]
pct_rise = total_rise / first["co2_level"] * 100
y_first, y_last = int(first["year"]), int(last["year"])

# ---- graph ----
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=g1["year"], y=g1["co2_level"], mode="lines", name="CO2 level",
    line=dict(color="#4C78A8", width=3),
    fill="tozeroy", fillcolor="rgba(76,120,168,0.10)",
    hovertemplate="<b>%{x}</b><br>%{y:.1f} ppm<extra></extra>",
))
for m in [350, 400]:
    reached = g1[g1["co2_level"] >= m]
    if reached.empty:
        continue
    row = reached.iloc[0]
    fig.add_trace(go.Scatter(
        x=[row["year"]], y=[row["co2_level"]], mode="markers+text",
        name=f"Passed {m} ppm",
        text=[f"{int(row['year'])}<br>{m} ppm"], textposition="top left",
        marker=dict(size=12, color="#E45756", line=dict(color="white", width=1)),
    ))
for r_, pos in [(first, "top center"), (last, "top center")]:
    fig.add_trace(go.Scatter(
        x=[r_["year"]], y=[r_["co2_level"]], mode="markers+text",
        text=[f"{r_['co2_level']:.0f} ppm"], textposition=pos,
        marker=dict(size=9, color="#333"), showlegend=False,
    ))
fig.update_layout(
    template="plotly_white", width=1090, height=480,
    title=dict(text=f"<b>Estimated Atmospheric CO2 Level, {y_first}-{y_last}</b>"
                    f"<br><sup>Up {total_rise:.0f} ppm ({pct_rise:.0f}%) since {y_first}</sup>", x=0.02),
    xaxis_title="Year", yaxis_title="CO2 concentration (ppm)",
    yaxis=dict(range=[290, 450]),
    legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
    margin=dict(t=90, b=100), hovermode="x unified",
)
fig.show()

# ---- summary ----
declines = int((g1["co2_ppm"] < 0).sum())
continuity = ("no year shows a decline" if declines == 0
              else f"{declines} year(s) show a fall in the growth figure")
summary = (
    f"Atmospheric CO2 rose from about {first['co2_level']:.1f} ppm in {y_first} to about "
    f"{last['co2_level']:.1f} ppm in {y_last}, an increase of roughly {total_rise:.1f} ppm "
    f"(about {pct_rise:.0f}%). The rise has been continuous: {continuity}."
)
print(summary)
save_report(TOPIC, "q1", "How has CO2 changed from 1960 to now?", fig, summary)

Atmospheric CO2 rose from about 316.9 ppm in 1960 to about 427.4 ppm in 2025, an increase of roughly 110.4 ppm (about 35%). The rise has been continuous: no year shows a decline.
[report] saved global_trends/q1 (1 graph(s))


### Q2. Is the growth rate of CO2 itself accelerating?

In [3]:
g1 = g1.sort_values("year").reset_index(drop=True)
lr = linregress(g1["year"], g1["co2_ppm"])
g1["trend"] = lr.intercept + lr.slope * g1["year"]
g1["roll5"] = g1["co2_ppm"].rolling(5, center=True, min_periods=3).mean()

# ---- graph ----
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=g1["year"], y=g1["co2_ppm"], mode="lines+markers",
    name="Yearly growth", line=dict(color="#9DB7D5", width=1.5), marker=dict(size=5),
    hovertemplate="<b>%{x}</b><br>%{y:.2f} ppm/yr<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=g1["year"], y=g1["roll5"], mode="lines", name="5-year rolling average",
    line=dict(color="#F58518", width=3),
))
fig.add_trace(go.Scatter(
    x=g1["year"], y=g1["trend"], mode="lines",
    name=f"Trend (+{lr.slope * 10:.2f} ppm/yr per decade)",
    line=dict(color="#E45756", width=2.5, dash="dash"),
))
fig.update_layout(
    template="plotly_white", width=1090, height=480,
    title=dict(text="<b>Is the CO2 Growth Rate Accelerating?</b>"
                    f"<br><sup>Trend slope: {lr.slope:.3f} ppm/yr each year "
                    f"(p = {lr.pvalue:.1e}, R2 = {lr.rvalue**2:.2f})</sup>", x=0.02),
    xaxis_title="Year", yaxis_title="CO2 growth (ppm per year)",
    yaxis=dict(range=[0, 4]),
    legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
    margin=dict(t=90, b=100), hovermode="x unified",
)
fig.show()

# Decade averages (less noisy than single years)
dec = g1.groupby(g1["year"] // 10 * 10)["co2_ppm"].mean().round(2)
print("Average growth per decade (ppm/yr):")
print(dec.to_string())

# ---- summary ----
accelerating = lr.slope > 0 and lr.pvalue < 0.05
summary = (
    f"{'Yes' if accelerating else 'No clear acceleration'}: the average annual growth rate of atmospheric CO2 "
    f"moved from {dec.iloc[0]:.2f} ppm/yr in the {dec.index[0]}s to {dec.iloc[-1]:.2f} ppm/yr in the "
    f"{dec.index[-1]}s (about {dec.iloc[-1] / dec.iloc[0]:.1f} times higher). A linear trend changes the growth "
    f"rate by {lr.slope:.3f} ppm/yr every year (about {lr.slope * 10:.2f} ppm/yr per decade, "
    f"R2 = {lr.rvalue**2:.2f}). Short-term spikes and dips are mainly driven by El Nino and volcanic events "
    f"(for example the 1991 Pinatubo eruption) on top of this long-term trend."
)
print(summary)
save_report(TOPIC, "q2", "Is the growth rate of CO2 itself accelerating?", fig, summary)

Average growth per decade (ppm/yr):
year
1960    0.86
1970    1.22
1980    1.64
1990    1.53
2000    1.91
2010    2.40
2020    2.62
Yes: the average annual growth rate of atmospheric CO2 moved from 0.86 ppm/yr in the 1960s to 2.62 ppm/yr in the 2020s (about 3.0 times higher). A linear trend changes the growth rate by 0.029 ppm/yr every year (about 0.29 ppm/yr per decade, R2 = 0.61). Short-term spikes and dips are mainly driven by El Nino and volcanic events (for example the 1991 Pinatubo eruption) on top of this long-term trend.
[report] saved global_trends/q2 (1 graph(s))


### Q3. What is the seasonal cycle of CO2 and which month peaks?

In [4]:
m = monthlyDf.sort_values(["year", "month"]).copy()

roll = m["co2_ppm"].rolling(12, center=True).mean()
m["trend"] = (roll + roll.shift(-1)) / 2          # properly centered 12-month trend

# Seasonal part = CO2 minus the trend
m["anomaly"] = m["co2_ppm"] - m["trend"]
m = m.dropna(subset=["anomaly"])

# Keep only years that still have all 12 months
cnt = m.groupby("year")["month"].nunique()
m = m[m["year"].isin(cnt[cnt == 12].index)].copy()

names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
seasonal = m.groupby("month")["anomaly"].agg(["median", "mean", "std", "count"]).reset_index()
seasonal["name"] = seasonal["month"].apply(lambda i: names[i - 1])

peak = seasonal.loc[seasonal["median"].idxmax()]
trough = seasonal.loc[seasonal["median"].idxmin()]
amplitude = peak["median"] - trough["median"]
n_years = m["year"].nunique()

# ---- graph ----
fig = go.Figure()
for i, name in enumerate(names, start=1):
    vals = m.loc[m["month"] == i, "anomaly"]
    fig.add_trace(go.Box(
        y=vals, x=[name] * len(vals), name=name,
        boxpoints="all", jitter=0.4, pointpos=0,
        marker=dict(size=4, opacity=0.5), line=dict(width=1.5),
        fillcolor="rgba(228,87,86,0.25)" if vals.median() >= 0 else "rgba(76,120,168,0.25)",
        marker_color="#E45756" if vals.median() >= 0 else "#4C78A8",
        showlegend=False,
        hovertemplate="%{y:+.2f} ppm<extra>" + name + "</extra>",
    ))
fig.add_trace(go.Scatter(
    x=seasonal["name"], y=seasonal["median"], mode="lines+markers",
    line=dict(color="#222", width=3), marker=dict(size=8), name="Median (central value)",
))
fig.add_trace(go.Scatter(
    x=[peak["name"]], y=[peak["median"]], mode="markers+text",
    marker=dict(color="red", size=15, line=dict(color="white", width=1.5)),
    text=[f"Peak: {peak['name']}<br>{peak['median']:+.2f}"], textposition="top center", name="Peak month",
))
fig.add_trace(go.Scatter(
    x=[trough["name"]], y=[trough["median"]], mode="markers+text",
    marker=dict(color="green", size=15, line=dict(color="white", width=1.5)),
    text=[f"Low: {trough['name']}<br>{trough['median']:+.2f}"], textposition="bottom center", name="Lowest month",
))
fig.update_layout(
    template="plotly_white", width=1090, height=520,
    title=dict(text="<b>Seasonal Cycle of CO2</b>"
                    f"<br><sup>{n_years} years | Peak in {peak['name']}, lowest in {trough['name']}, "
                    f"swing of {amplitude:.1f} ppm</sup>", x=0.02),
    xaxis=dict(title="Month", categoryorder="array", categoryarray=names),
    yaxis=dict(title="CO2 vs 12-month trend (ppm)"),
    legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
    margin=dict(t=90, b=100),
)
fig.add_hline(y=0, line_color="gray", line_dash="dash")
fig.show()

out = seasonal[["name", "median", "mean", "std", "count"]].round(2)
out.columns = ["Month", "Median", "Mean", "Std", "Years"]
print(out.to_string(index=False))

# ---- summary ----
top2 = seasonal.nlargest(2, "median").reset_index(drop=True)
low2 = seasonal.nsmallest(2, "median").reset_index(drop=True)
summary = (
    f"Atmospheric CO2 follows a regular seasonal cycle. After removing the long-term trend with a 12-month "
    f"moving average, the median across {n_years} complete years shows CO2 peaking in {top2.loc[0, 'name']} "
    f"({top2.loc[0, 'median']:+.2f} ppm above trend), with {top2.loc[1, 'name']} close behind "
    f"({top2.loc[1, 'median']:+.2f} ppm). It then falls through the summer to its lowest point in "
    f"{low2.loc[0, 'name']} ({low2.loc[0, 'median']:+.2f} ppm), with {low2.loc[1, 'name']} almost equal "
    f"({low2.loc[1, 'median']:+.2f} ppm). The seasonal swing is about {amplitude:.2f} ppm from peak to trough. "
    f"The cycle arises because plants absorb CO2 during the summer growing season and release it through "
    f"decay in autumn and winter. This swing is small compared with the long-term rise of "
    f"{total_rise:.0f} ppm since {y_first}."
)
print(summary)
save_report(TOPIC, "q3", "What is the seasonal cycle of CO2 and which month peaks?", fig, summary)

Month  Median  Mean  Std  Years
  Jan    0.96  0.93 0.20     46
  Feb    1.27  1.25 0.21     46
  Mar    1.54  1.52 0.19     46
  Apr    1.80  1.76 0.15     46
  May    1.61  1.61 0.11     46
  Jun    0.59  0.61 0.19     46
  Jul   -1.16 -1.16 0.29     46
  Aug   -2.62 -2.61 0.31     46
  Sep   -2.65 -2.66 0.22     46
  Oct   -1.48 -1.47 0.15     46
  Nov   -0.27 -0.26 0.16     46
  Dec    0.48  0.48 0.19     46
Atmospheric CO2 follows a regular seasonal cycle. After removing the long-term trend with a 12-month moving average, the median across 46 complete years shows CO2 peaking in Apr (+1.80 ppm above trend), with May close behind (+1.61 ppm). It then falls through the summer to its lowest point in Sep (-2.65 ppm), with Aug almost equal (-2.62 ppm). The seasonal swing is about 4.45 ppm from peak to trough. The cycle arises because plants absorb CO2 during the summer growing season and release it through decay in autumn and winter. This swing is small compared with the long-term rise 

### Q4. How have CH4 and N2O changed, and are they growing faster or slower than CO2 in relative terms?

In [5]:
gases = ["co2_ppm", "ch4_ppb", "n2o_ppb"]
labels = {"co2_ppm": "CO2", "ch4_ppb": "CH4", "n2o_ppb": "N2O"}
colors = {"co2_ppm": "#3B82F6", "ch4_ppb": "#EF4444", "n2o_ppb": "#10B981"}

# Keep years where all three gases have data
d = yearlyDf.dropna(subset=gases).sort_values("year").reset_index(drop=True).copy()
if len(d) < 2:
    raise ValueError("At least two complete years are required.")

start, end = int(d["year"].iloc[0]), int(d["year"].iloc[-1])
years = end - start

# Relative change from the common baseline
for g in gases:
    d[g + "_pct"] = (d[g] / d[g].iloc[0] - 1) * 100

# ---- graph ----
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
    subplot_titles=("Relative growth since the baseline year",
                    "Annual change in relative growth (percentage points)"),
)
for g in gases:
    pct_col = g + "_pct"
    fig.add_trace(go.Scatter(
        x=d["year"], y=d[pct_col], mode="lines", name=labels[g],
        line=dict(color=colors[g], width=3),
        hovertemplate="<b>%{x}</b><br>" + labels[g] + ": %{y:.2f}%<extra></extra>",
    ), row=1, col=1)
    final_pct = d[pct_col].iloc[-1]
    fig.add_trace(go.Scatter(
        x=[end], y=[final_pct], mode="markers+text", text=[f"{final_pct:+.1f}%"],
        textposition="middle right", textfont=dict(color=colors[g], size=12),
        marker=dict(color=colors[g], size=8), showlegend=False, hoverinfo="skip", cliponaxis=False,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=d["year"], y=d[pct_col].diff(), mode="lines", name=labels[g] + " annual change",
        line=dict(color=colors[g], width=1.8), showlegend=False,
        hovertemplate="<b>%{x}</b><br>" + labels[g] + ": %{y:+.3f} pp<extra></extra>",
    ), row=2, col=1)
fig.update_layout(
    template="plotly_white", width=1090, height=780, hovermode="x unified",
    title=dict(text=f"<b>How Have CO2, CH4 and N2O Changed?</b>"
                    f"<br><sup>Relative change from {start} to {end} | common baseline for all gases</sup>",
               x=0.03, xanchor="left"),
    legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0),
    margin=dict(l=75, r=110, t=125, b=65),
)
fig.update_yaxes(title_text="Change since baseline (%)", zeroline=True, zerolinecolor="#9CA3AF", row=1, col=1)
fig.update_yaxes(title_text="Percentage points per year", zeroline=True, zerolinecolor="#9CA3AF", row=2, col=1)
fig.update_xaxes(title_text="Year", row=2, col=1)
fig.show()

# ---- statistics ----
rows = []
for g in gases:
    initial, final = d[g].iloc[0], d[g].iloc[-1]
    rows.append({
        "Gas": labels[g],
        "Start concentration": initial,
        "End concentration": final,
        "Absolute change": final - initial,
        "Total change (%)": (final / initial - 1) * 100,
        "Annualized growth (%)": ((final / initial) ** (1 / years) - 1) * 100,
        "Absolute change/year": (final - initial) / years,
    })
stats = pd.DataFrame(rows).set_index("Gas")
print(f"Period: {start}-{end} ({years} elapsed years)")
print(stats.round(3).to_string())

# ---- summary ----
co2_rate = stats.loc["CO2", "Annualized growth (%)"]

def compare(gas):
    diff = stats.loc[gas, "Annualized growth (%)"] - co2_rate
    return "faster than" if diff > 0.001 else "slower than" if diff < -0.001 else "at about the same rate as"

fastest = stats["Annualized growth (%)"].idxmax()
summary = (
    f"From {start} to {end}, atmospheric concentrations of all three greenhouse gases increased, and {fastest} "
    f"showed the highest relative growth. CO2 rose {stats.loc['CO2', 'Total change (%)']:.2f}% overall, "
    f"compared with {stats.loc['CH4', 'Total change (%)']:.2f}% for CH4 and "
    f"{stats.loc['N2O', 'Total change (%)']:.2f}% for N2O. On an annualized basis, CO2 grew at {co2_rate:.3f}% "
    f"per year, CH4 at {stats.loc['CH4', 'Annualized growth (%)']:.3f}% and N2O at "
    f"{stats.loc['N2O', 'Annualized growth (%)']:.3f}%. In percentage terms, CH4 accumulated {compare('CH4')} CO2 "
    f"and N2O accumulated {compare('N2O')} CO2."
)
print(summary)
save_report(TOPIC, "q4",
            "How have CH4 and N2O changed, and are they growing faster or slower than CO2 in relative terms?",
            fig, summary)

Period: 2001-2025 (24 elapsed years)
     Start concentration  End concentration  Absolute change  Total change (%)  Annualized growth (%)  Absolute change/year
Gas                                                                                                                        
CO2              370.566            425.622           55.057            14.857                  0.579                 2.294
CH4             1771.320           1935.990          164.670             9.296                  0.371                 6.861
N2O              316.364            338.864           22.500             7.112                  0.287                 0.938
From 2001 to 2025, atmospheric concentrations of all three greenhouse gases increased, and CO2 showed the highest relative growth. CO2 rose 14.86% overall, compared with 9.30% for CH4 and 7.11% for N2O. On an annualized basis, CO2 grew at 0.579% per year, CH4 at 0.371% and N2O at 0.287%. In percentage terms, CH4 accumulated slower than CO2 a

### Q5. How did global temperature anomaly change by decade, and which decade jumped most?

In [6]:
t1 = (yearlyDf[["year", "temp_anomaly"]].dropna().sort_values("year").reset_index(drop=True).copy())
t1["year"] = t1["year"].astype(int)
t1["decade"] = (t1["year"] // 10) * 10

decadal = (
    t1.groupby("decade")
    .agg(mean_anomaly=("temp_anomaly", "mean"), min_anomaly=("temp_anomaly", "min"),
         max_anomaly=("temp_anomaly", "max"), years_available=("temp_anomaly", "count"))
    .reset_index()
)
start_year, end_year = int(t1["year"].min()), int(t1["year"].max())

# Linear trend across decade averages
x, y = decadal["decade"].to_numpy(), decadal["mean_anomaly"].to_numpy()
slope, intercept = np.polyfit(x, y, 1)
decadal["trend"] = slope * x + intercept
decadal["jump"] = decadal["mean_anomaly"].diff()

# ---- graph ----
fig = go.Figure()
fig.add_trace(go.Scatter(x=decadal["decade"], y=decadal["max_anomaly"], mode="lines",
                         line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=decadal["decade"], y=decadal["min_anomaly"], mode="lines", fill="tonexty",
                         fillcolor="rgba(76, 120, 168, 0.12)", line=dict(width=0),
                         name="Yearly range within decade", hoverinfo="skip"))
fig.add_trace(go.Scatter(
    x=decadal["decade"], y=decadal["mean_anomaly"], mode="lines+markers", name="Decade average",
    line=dict(color="#4C78A8", width=3), marker=dict(size=7),
    customdata=decadal[["years_available"]],
    hovertemplate="<b>%{x}s</b><br>Mean anomaly: %{y:+.2f} °C<br>Years available: %{customdata[0]}<extra></extra>",
))
fig.add_trace(go.Scatter(x=decadal["decade"], y=decadal["trend"], mode="lines", name="Linear trend",
                         line=dict(color="#E45756", width=2, dash="dash"),
                         hovertemplate="Trend: %{y:+.2f} °C<extra></extra>"))
fig.add_hline(y=0, line_dash="dot", line_color="#777777", annotation_text="Zero anomaly",
              annotation_position="bottom right")
fig.update_layout(
    template="plotly_white", width=1090, height=560, hovermode="x unified",
    title=dict(text="<b>Long-Term Change in Global Surface Temperature</b>"
                    f"<br><sup>Decade averages | {start_year}-{end_year}</sup>", x=0.03, xanchor="left"),
    xaxis_title="Decade", yaxis_title="Temperature anomaly (°C)",
    margin=dict(l=75, r=40, t=110, b=80),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.update_xaxes(dtick=10)
fig.update_yaxes(zeroline=False, ticksuffix="°C")
fig.show()

print(f"Period: {start_year}-{end_year} | decades: {len(decadal)}")
print(decadal[["decade", "mean_anomaly", "jump", "years_available"]].round(3).to_string(index=False))

# ---- summary ----
first_dec, latest_dec = decadal.iloc[0], decadal.iloc[-1]
change = latest_dec["mean_anomaly"] - first_dec["mean_anomaly"]
jmp = decadal.loc[decadal["jump"].idxmax()]
partial = (f" Note that the {int(latest_dec['decade'])}s cover only {int(latest_dec['years_available'])} years "
           f"so far, so that decade is incomplete.") if latest_dec["years_available"] < 10 else ""
summary = (
    f"Global temperature anomalies show a clear long-term warming trend, rising from an average of "
    f"{first_dec['mean_anomaly']:+.3f} °C in the {int(first_dec['decade'])}s to "
    f"{latest_dec['mean_anomaly']:+.3f} °C in the {int(latest_dec['decade'])}s, a difference of {change:.3f} °C "
    f"between the two decade averages. The largest increase between consecutive decade averages occurred from "
    f"the {int(jmp['decade']) - 10}s to the {int(jmp['decade'])}s, rising by {jmp['jump']:.3f} °C.{partial} "
    f"The linear trend across decade averages is {slope * 10:+.3f} °C per decade."
)
print(summary)
save_report(TOPIC, "q5", "How did global temperature anomaly change by decade, and which decade jumped most?",
            fig, summary)

Period: 1880-2025 | decades: 15
 decade  mean_anomaly   jump  years_available
   1880        -0.217    NaN               10
   1890        -0.242 -0.025               10
   1900        -0.320 -0.078               10
   1910        -0.335 -0.015               10
   1920        -0.240  0.095               10
   1930        -0.125  0.115               10
   1940         0.043  0.167               10
   1950        -0.048 -0.090               10
   1960        -0.030  0.018               10
   1970         0.034  0.065               10
   1980         0.247  0.213               10
   1990         0.387  0.140               10
   2000         0.589  0.202               10
   2010         0.806  0.217               10
   2020         1.067  0.261                6
Global temperature anomalies show a clear long-term warming trend, rising from an average of -0.217 °C in the 1880s to +1.067 °C in the 2020s, a difference of 1.284 °C between the two decade averages. The largest increase between co